# Escalações e Copas

## Task 4: conferir as escalações, trazer os dados de cada Copa e levar o código para o projeto

## Os três arquivos juntos

- `WorldCupMatches.csv`: uma linha por partida
- `WorldCupPlayers.csv`: uma linha por jogador em cada partida
- `WorldCups.csv`: uma linha por Copa, com sede e campeão

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-joins")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: o mesmo código dos vídeos 2.1 e 2.2.

In [ ]:
# vídeo 2.1: leitura e limpeza
from wc_pipeline.schemas import MATCHES

partidas_brutas = (
    spark.read
    .option("header", "true")
    .schema(MATCHES)
    .csv(f"{RAW}/WorldCupMatches.csv")
)

partidas_limpas = (
    partidas_brutas
    .filter(F.col("MatchID").isNotNull())
    .dropDuplicates(["MatchID"])
)

# vídeo 2.2: padronização
jogos = partidas_limpas.select(
    F.col("Year").alias("ano"),
    F.col("MatchID").alias("match_id"),
    F.col("Stage").alias("fase"),
    F.col("Home Team Name").alias("mandante"),
    F.col("Home Team Goals").alias("gols_mandante"),
    F.col("Away Team Goals").alias("gols_visitante"),
    F.col("Away Team Name").alias("visitante"),
    F.col("Attendance").alias("publico"),
    F.col("Datetime").alias("data_partida"),
)

jogos = jogos.withColumn(
    "publico",
    F.regexp_replace("publico", ",", "").cast("int"),
)

FORMATO_DATA = "dd MMM yyyy - HH:mm"

jogos = jogos.withColumn(
    "data_partida",
    F.to_timestamp("data_partida", FORMATO_DATA),
)

partidas = (
    jogos
    .withColumn("total_gols", F.col("gols_mandante") + F.col("gols_visitante"))
    .withColumn(
        "resultado",
        F.when(F.col("gols_mandante") > F.col("gols_visitante"), "mandante")
         .when(F.col("gols_mandante") < F.col("gols_visitante"), "visitante")
         .otherwise("empate"),
    )
)

partidas.printSchema()

## Os tipos de join

| Tipo | O que devolve | Pergunta |
|---|---|---|
| `inner` | só as linhas com par nos dois lados | "partidas **com** escalações" |
| `left` | tudo de A; colunas de B `null` sem par | "todas as partidas, com a Copa **se houver**" |
| `left_anti` | linhas de A **sem** par em B | "quais partidas **não têm** escalação?" (`NOT EXISTS`) |
| `left_semi` | linhas de A **com** par em B | "quais partidas **têm** escalação?" (`EXISTS`) |

## left_anti: as escalações conversam com as partidas?

Partidas sem escalação, e escalações sem partida:

A demanda não pede estatísticas com as escalações: o arquivo foi conferido e não entra nas próximas tasks.

## left join: os dados de cada Copa

Cada partida casa com exatamente uma Copa: continua uma linha por partida.

## Levando o código para o projeto

> Notebook é para explorar. A pasta `src/` é o que vai para produção.

`%%writefile` grava o conteúdo da célula num arquivo. Em `src/wc_pipeline/transformacoes.py`:

- `limpar_partidas`: limpeza e padronização (tasks 1 e 2)
- `partidas_por_selecao`: cada partida vira duas linhas, com V, E ou D (task 3)
- `resumo_por_selecao`: as métricas de cada seleção em cada Copa

- Cada função **recebe um DataFrame e devolve outro**: serve no notebook e no pipeline (aula 5).
- `limpar_partidas` = limpeza (task 1) + padronização (task 2), com estádio, cidade e condição de vitória.

## Demanda da aula 2 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | partidas auditadas e limpas |
| Task 2 | nomes em português e tipos corrigidos |
| Task 3 | tabelas por Copa e por seleção, conferidas pelo total de gols |
| Task 4 | escalações conferidas, dados de cada Copa nas partidas, e a limpeza em `src/wc_pipeline/transformacoes.py` |

## Para consultar depois

- **Join que multiplica linhas:** cada partida tem 22 jogadores nas escalações, então um `inner` join de partidas com
  escalações repete cada partida 22 vezes. Somar `total_gols` depois disso multiplica os gols por 22. Antes de agregar
  depois de um join, pergunte qual é o grão: o que cada linha representa agora?
- **Chaves com nomes diferentes:** quando a coluna não tem o mesmo nome nos dois lados, use uma expressão, por
  exemplo `a.join(b, a.ano == b.Year, "left")`. Com uma expressão, o resultado fica com as duas colunas da chave;
  com `on="nome"`, com uma só.
- **`left_semi`:** devolve as linhas da esquerda que **têm** par na direita, só com as colunas da esquerda. É o
  `WHERE EXISTS` do SQL, o contrário do `left_anti`.
- **Joins com uma tabela pequena** (como as 18 Copas) têm um truque de desempenho, o broadcast join, que fica para a
  aula 4.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()